<a href="https://colab.research.google.com/github/rameenhamad/Certura_ML_Internship_tasks/blob/main/1_House_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Important Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split

loading dataset

In [ ]:
#boston housing data set was not loading due to some investigation error
dt = datasets.fetch_california_housing()

making dataframe of dataset

In [ ]:
df = pd.DataFrame(data= dt.data, columns=dt.feature_names)
target_column = dt.target_names[0]
df[target_column] = dt.target
df.head(5)

In [ ]:
df.drop_duplicates()
df.isnull().sum()
df.shape
df.info()
df.describe()


data distribution in dataset

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as st

n_features = df.shape[1]
fig, axes = plt.subplots(n_features, 2, figsize=(10, 4*n_features))

for i, col in enumerate(df.columns):
    # Histogram
    axes[i, 0].hist(df[col], bins=30, alpha=0.7)
    axes[i, 0].set_title(f"Histogram of {col}")

    # QQ plot
    st.probplot(df[col].values, plot=axes[i, 1])
    axes[i, 1].set_title(f"QQ Plot of {col}")

plt.tight_layout()
plt.show()

feature relation to Y

In [ ]:
X = df.drop('MedHouseVal', axis=1)
Y = df['MedHouseVal']

train_test_spliting data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=37)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

sekewed y

In [ ]:
import scipy.stats as st

st.probplot(Y, plot = plt)

plt.show()

Feature transformation

In [ ]:
# from sklearn.preprocessing import PowerTransformer

# pt = PowerTransformer()
# X_train_trans = pt.fit_transform(X_train)
# X_test_trans = pt.transform(X_test)

input features after transform

In [ ]:
# n_features = X_train_trans.shape[1]
# feature_names = X.columns  # Saved before transformation

# fig, axes = plt.subplots(n_features, 2, figsize=(10, 4*n_features))

# for i in range(n_features):
#     # Histogram
#     axes[i, 0].hist(X_train_trans[:, i], bins=30, alpha=0.7)
#     axes[i, 0].set_title(f"Histogram of {feature_names[i]}")

#     # QQ plot
#     st.probplot(X_train_trans[:, i], plot=axes[i, 1])
#     axes[i, 1].set_title(f"QQ Plot of {feature_names[i]}")

# plt.tight_layout()
# plt.show()

target transformation as data highly sekewed

In [ ]:
from sklearn.preprocessing import FunctionTransformer

ft = FunctionTransformer(func=np.log1p)

Y_train_trans = ft.fit_transform(Y_train)
Y_test_trans = ft.transform(Y_test)


plynomial feature transformer as non linearity of dataset

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

pf = PolynomialFeatures(degree=2, interaction_only=True)
X_train_pf = pf.fit_transform(X_train)
X_test_pf = pf.transform(X_test)

Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_pf)
X_test_scaled = scaler.transform(X_test_pf)

# X_train_scaled = robust_scaler.fit_transform(X_train)
# X_test_scaled = robust_scaler.transform(X_test)

y after transform

In [ ]:
import scipy.stats as st

st.probplot(Y_test_trans, plot = plt)

plt.show()

Model training

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled, Y_train_trans)
Y_predict = model.predict(X_test_scaled)

#getting inverse of Y to actual value from log
Y_pred_norm = np.expm1(Y_predict)

Root Mean Squared Error

In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(Y_test, Y_pred_norm)
rmse_log = root_mean_squared_error(Y_test_trans, Y_predict)
print(f"Root Mean Squared Error (RMSE) for Raw values(Original Y): {rmse}")
print(f"Root Mean Squared Error (RMSE) Log Transformed Y : {rmse_log}")

Plot predictions vs actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(Y_test, Y_pred_norm, alpha=0.6)
axes[0].plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--')
axes[0].set_xlabel("Actual House Prices")
axes[0].set_ylabel("Predicted House Prices")
axes[0].set_title("Actual vs Predicted (Original Y)")

axes[1].scatter(Y_test_trans, Y_predict, alpha=0.6)
axes[1].plot([Y_test_trans.min(), Y_test_trans.max()], [Y_test_trans.min(), Y_test_trans.max()], 'r--')
axes[1].set_xlabel("Actual House Prices (Transformed)")
axes[1].set_ylabel("Predicted House Prices")
axes[1].set_title("Actual vs Predicted (Transformed Y)")

plt.tight_layout()
plt.show()